In [10]:
# finetune_resnet50.py
import torch, torch.nn as nn, torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

train_dir = "dataset_augmented/train"
val_dir   = "dataset/test"
batch_size = 16
epochs = 10
num_classes = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
train_ds = datasets.ImageFolder(train_dir, transform=transform)
val_ds   = datasets.ImageFolder(val_dir, transform=transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

# ResNet50
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
for param in resnet.parameters():
    param.requires_grad = False
for name, param in resnet.layer4.named_parameters():
    param.requires_grad = True

num_ftrs = resnet.fc.in_features
resnet.fc = nn.Linear(num_ftrs, num_classes)
resnet = resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, resnet.parameters()), lr=1e-4)

# Training loop
for epoch in range(epochs):
    resnet.train()
    correct, total = 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = resnet(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    print(f"Epoch {epoch+1}: Train Acc = {correct/total:.4f}")

# Save fine-tuned model
torch.save(resnet.state_dict(), "resnet50_finetuned.pth")
print("✅ Saved fine-tuned ResNet50")


Epoch 1: 100%|███████████████████████████████████████████████████████████████████████| 358/358 [02:28<00:00,  2.41it/s]


Epoch 1: Train Acc = 0.6966


Epoch 2: 100%|███████████████████████████████████████████████████████████████████████| 358/358 [02:26<00:00,  2.45it/s]


Epoch 2: Train Acc = 0.7433


Epoch 3: 100%|███████████████████████████████████████████████████████████████████████| 358/358 [02:31<00:00,  2.36it/s]


...

Epoch 8: Train Acc = 0.9736


Epoch 9: 100%|███████████████████████████████████████████████████████████████████████| 358/358 [02:25<00:00,  2.47it/s]


Epoch 9: Train Acc = 0.9806


Epoch 10: 100%|██████████████████████████████████████████████████████████████████████| 358/358 [02:23<00:00,  2.50it/s]

Epoch 10: Train Acc = 0.9797
✅ Saved fine-tuned ResNet50


In [16]:
# use_finetuned_resnet_qsvm.py

import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import numpy as np
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, classification_report, confusion_matrix
import pennylane as qml
from collections import Counter

# ---------------------------
# Config
# ---------------------------
train_dir = "dataset_augmented/train"
test_dir  = "dataset/test"
batch_size = 32
num_classes = 2
n_qubits = 12   # also PCA components
cnn_weights_path = "resnet50_finetuned.pth"  # your trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# Data
# ---------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
train_ds = datasets.ImageFolder(train_dir, transform=transform)
test_ds  = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

print("Train dist:", Counter(train_ds.targets))
print("Test dist:", Counter(test_ds.targets))

# ---------------------------
# Load fine-tuned ResNet backbone (remove classifier)
# ---------------------------
resnet = models.resnet50(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)  # dummy head
resnet.load_state_dict(torch.load(cnn_weights_path, map_location=device))
modules = list(resnet.children())[:-1]  # drop fc
feat_extractor = nn.Sequential(*modules).to(device)
feat_extractor.eval()

def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for imgs, labs in loader:
            imgs = imgs.to(device)
            f = feat_extractor(imgs).view(imgs.size(0), -1)  # [B,2048]
            feats.append(f.cpu().numpy())
            labels.append(labs.numpy())
    return np.vstack(feats), np.hstack(labels)

X_train, y_train = extract_features(train_loader)
X_test,  y_test  = extract_features(test_loader)
print("Feature shapes:", X_train.shape, X_test.shape)

# ---------------------------
# PCA compression
# ---------------------------
pca = PCA(n_components=n_qubits, random_state=42)
pca.fit(X_train)
Xtr = pca.transform(X_train)
Xte = pca.transform(X_test)

# Scale to angles [-π/2, π/2]
mins, maxs = Xtr.min(axis=0), Xtr.max(axis=0)
ranges = np.where(maxs-mins == 0, 1, maxs-mins)
def scale_to_angles(X):
    Xn = (X - mins) / ranges
    return (Xn - 0.5) * np.pi
Xtr_angles, Xte_angles = scale_to_angles(Xtr), scale_to_angles(Xte)

# ---------------------------
# Quantum kernel
# ---------------------------
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="autograd")
def qstate(x):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    for i in range(n_qubits-1):
        qml.CNOT(wires=[i, i+1])  # entangler
    return qml.state()

def compute_states(X_angles):
    return np.vstack([np.array(qstate(x)) for x in X_angles])

print("Computing quantum states...")
states_train = compute_states(Xtr_angles)
states_test  = compute_states(Xte_angles)

def kernel_from_states(A, B):
    inner = np.dot(A.conj(), B.T)
    return np.abs(inner)**2

K_train = kernel_from_states(states_train, states_train)
K_test  = kernel_from_states(states_test, states_train)

# ---------------------------
# Train QSVM
# ---------------------------

svc = SVC(kernel="precomputed", class_weight="balanced", random_state=42, probability=True)
svc.fit(K_train, y_train)
y_pred = svc.predict(K_test)

# ---------------------------
# Evaluation
# ---------------------------
acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
minority_recall = recall_score(y_test, y_pred, pos_label=1)

print(f"\n✅ QSVM Accuracy: {acc:.4f}")
print(f"Balanced Acc: {bal_acc:.4f}, Minority Recall: {minority_recall:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=test_ds.classes))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Train dist: Counter({0: 4060, 1: 1659})
Test dist: Counter({0: 144, 1: 59})
Feature shapes: (5719, 2048) (203, 2048)
Computing quantum states...

✅ QSVM Accuracy: 0.6502
Balanced Acc: 0.5284, Minority Recall: 0.2373

Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.82      0.77       144
           1       0.35      0.24      0.28        59

    accuracy                           0.65       203
   macro avg       0.54      0.53      0.53       203
weighted avg       0.62      0.65      0.63       203

Confusion Matrix:
 [[118  26]
 [ 45  14]]


In [17]:
import joblib
import torch

# Save using joblib (preferred for sklearn models)
joblib.dump(svc, "qsvm_model.pkl")
print("QSVM model saved as qsvm_model.pkl")

# Alternatively, save with torch (still works but less common for sklearn)
torch.save(svc, "qsvm_model_main.pth")
print("QSVM model also saved as qsvm_model_main.pth")


QSVM model saved as qsvm_model.pkl
QSVM model also saved as qsvm_model_main.pth


In [14]:
# cnn_features_svm.py
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import numpy as np
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, classification_report, confusion_matrix
from collections import Counter

# ---------------------------
# Config
# ---------------------------
train_dir = "dataset_augmented/train"
test_dir  = "dataset/test"
batch_size = 32
num_classes = 2
n_components = 12   # PCA components (same as qubits in QSVM)
cnn_weights_path = "resnet50_finetuned.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# Data
# ---------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
train_ds = datasets.ImageFolder(train_dir, transform=transform)
test_ds  = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

print("Train dist:", Counter(train_ds.targets))
print("Test dist:", Counter(test_ds.targets))

# ---------------------------
# Load fine-tuned ResNet backbone
# ---------------------------
resnet = models.resnet50(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet.load_state_dict(torch.load(cnn_weights_path, map_location=device))
modules = list(resnet.children())[:-1]  # remove FC
feat_extractor = nn.Sequential(*modules).to(device)
feat_extractor.eval()

def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for imgs, labs in loader:
            imgs = imgs.to(device)
            f = feat_extractor(imgs).view(imgs.size(0), -1)  # [B,2048]
            feats.append(f.cpu().numpy())
            labels.append(labs.numpy())
    return np.vstack(feats), np.hstack(labels)

X_train, y_train = extract_features(train_loader)
X_test,  y_test  = extract_features(test_loader)
print("Feature shapes:", X_train.shape, X_test.shape)

# ---------------------------
# PCA compression (optional, keeps comparability with QSVM)
# ---------------------------
pca = PCA(n_components=n_components, random_state=42)
pca.fit(X_train)
Xtr = pca.transform(X_train)
Xte = pca.transform(X_test)

# ---------------------------
# Classical SVM
# ---------------------------
svc = SVC(kernel="rbf", class_weight="balanced", random_state=42)
svc.fit(Xtr, y_train)
y_pred = svc.predict(Xte)

# ---------------------------
# Evaluation
# ---------------------------
acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
minority_recall = recall_score(y_test, y_pred, pos_label=1)

print(f"\n✅ Classical SVM Accuracy: {acc:.4f}")
print(f"Balanced Acc: {bal_acc:.4f}, Minority Recall: {minority_recall:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=test_ds.classes))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Train dist: Counter({0: 4060, 1: 1659})
Test dist: Counter({0: 144, 1: 59})
Feature shapes: (5719, 2048) (203, 2048)

✅ Classical SVM Accuracy: 0.6404
Balanced Acc: 0.5214, Minority Recall: 0.2373

Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.81      0.76       144
           1       0.33      0.24      0.28        59

    accuracy                           0.64       203
   macro avg       0.53      0.52      0.52       203
weighted avg       0.61      0.64      0.62       203

Confusion Matrix:
 [[116  28]
 [ 45  14]]
